# Experimento 3: Dataset MNIST

## Motor de Redes Neuronales - Optimización y Heurística
**Autores:** Raúl Mendoza, Adrián Ojeda, Varela  
**Universidad de Las Palmas de Gran Canaria**

---

## 1. Introducción

El dataset **MNIST** (Modified National Institute of Standards and Technology) es el benchmark más utilizado en deep learning para clasificación de imágenes. Contiene dígitos manuscritos del 0 al 9.

### Características del dataset:
| Propiedad | Valor |
|-----------|-------|
| Muestras de entrenamiento | 60,000 |
| Muestras de test | 10,000 |
| Tamaño de imagen | 28×28 píxeles |
| Canales | 1 (escala de grises) |
| Número de clases | 10 (dígitos 0-9) |
| Features (aplanado) | 784 |

Este experimento es **requisito obligatorio** para validar que el motor puede manejar datasets de gran escala y obtener resultados aceptables (≥80% accuracy).

## 2. Importación de Módulos

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import time

from src.network import NeuralNetwork
from src.layers import Dense
from src.optimizers import Adam
from src.trainer import Trainer
from src.data_utils import load_mnist, to_one_hot, train_val_test_split

np.random.seed(42)

print("Módulos importados correctamente ✓")

## 3. Carga y Exploración del Dataset MNIST

Descargamos el dataset MNIST automáticamente si no está presente. Los datos se normalizan al rango [0, 1] y se aplanan a vectores de 784 dimensiones.

In [ ]:
# Cargar MNIST (descarga automática si no existe)
X_train_full, y_train_full, X_test, y_test = load_mnist(
    data_dir="../data/mnist",
    normalize=True,
    flatten=True
)

print(f"\nDimensiones del dataset:")
print(f"  X_train: {X_train_full.shape}")
print(f"  y_train: {y_train_full.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_test:  {y_test.shape}")

print(f"\nRango de valores: [{X_train_full.min():.2f}, {X_train_full.max():.2f}]")
print(f"Clases: {np.unique(y_train_full)}")

## 4. Visualización de Ejemplos

Mostramos algunos ejemplos del dataset para entender mejor los datos.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('Ejemplos del dataset MNIST', fontsize=14)

for i in range(20):
    ax = axes[i // 10, i % 10]
    img = X_train_full[i].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{y_train_full[i]}')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../memoria/fig_mnist_examples.png', dpi=150)
plt.close()

print("Gráfica guardada en: memoria/fig_mnist_examples.png")
print("\nDistribución de clases en entrenamiento:")
for digit in range(10):
    count = np.sum(y_train_full == digit)
    print(f"  Dígito {digit}: {count:5d} muestras ({count/len(y_train_full)*100:.1f}%)")

## 5. Preprocesamiento de Datos

Preparamos los datos para el entrenamiento:
1. **Submuestra** (opcional): Para acelerar el entrenamiento inicial
2. **One-hot encoding**: Convertir etiquetas a vectores
3. **División train/val**: Separar conjunto de validación del entrenamiento

In [ ]:
y_train_onehot = to_one_hot(y_train_full, num_classes=10)
y_test_onehot = to_one_hot(y_test, num_classes=10)

n_train = int(0.85 * len(X_train_full))
indices = np.random.permutation(len(X_train_full))

train_idx = indices[:n_train]
val_idx = indices[n_train:]

X_train = X_train_full[train_idx]
y_train = y_train_onehot[train_idx]
X_val = X_train_full[val_idx]
y_val = y_train_onehot[val_idx]

print("Preprocesamiento completado:")
print(f"  Conjunto de entrenamiento: {X_train.shape[0]:,} muestras")
print(f"  Conjunto de validación:    {X_val.shape[0]:,} muestras")
print(f"  Conjunto de test:          {X_test.shape[0]:,} muestras")
print(f"\nForma de X_train: {X_train.shape}")
print(f"Forma de y_train: {y_train.shape}")

## 6. Definición de la Arquitectura de Red

Utilizamos una red con **dos capas ocultas** para obtener suficiente capacidad de aprendizaje.

**Arquitectura:**
- **Entrada:** 784 neuronas (28×28 píxeles aplanados)
- **Capa oculta 1:** 128 neuronas con ReLU
- **Capa oculta 2:** 64 neuronas con ReLU
- **Salida:** 10 neuronas con Softmax (10 dígitos)

In [ ]:
net = NeuralNetwork()

net.add(Dense(n_in=784, n_out=128, activation="relu", weight_init="he"))

net.add(Dense(n_in=128, n_out=64, activation="relu", weight_init="he"))

net.add(Dense(n_in=64, n_out=10, activation="softmax", weight_init="xavier"))

total_params = sum(p.size for p in net.params())

print("Arquitectura de la red:")
print("="*50)
print("  Entrada:    784 neuronas (28×28 píxeles)")
print("  Oculta 1:   128 neuronas (ReLU, He init)")
print("  Oculta 2:    64 neuronas (ReLU, He init)")
print("  Salida:      10 neuronas (Softmax, Xavier init)")
print("="*50)
print(f"Total parámetros: {total_params:,}")
print("")
print("Desglose de parámetros:")
print(f"  Capa 1: W(784×128) + b(128) = {784*128 + 128:,}")
print(f"  Capa 2: W(128×64) + b(64)   = {128*64 + 64:,}")
print(f"  Capa 3: W(64×10) + b(10)    = {64*10 + 10:,}")

## 7. Configuración del Entrenamiento

In [ ]:
LEARNING_RATE = 0.001
EPOCHS = 20
BATCH_SIZE = 64

optimizer = Adam(lr=LEARNING_RATE)
trainer = Trainer(net, optimizer, loss_name="cross_entropy")

print("Configuración del entrenamiento:")
print("="*50)
print(f"  Optimizador:    Adam")
print(f"  Learning Rate:  {LEARNING_RATE}")
print(f"  Pérdida:        Cross-Entropy")
print(f"  Épocas:         {EPOCHS}")
print(f"  Batch Size:     {BATCH_SIZE}")
print(f"  Batches/época:  {len(X_train) // BATCH_SIZE}")

## 8. Entrenamiento de la Red

Ejecutamos el bucle de entrenamiento y registramos las métricas.

In [ ]:
print("\n" + "="*60)
print("INICIANDO ENTRENAMIENTO")
print("="*60 + "\n")

start_time = time.time()

train_losses, val_losses = trainer.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=True
)

elapsed_time = time.time() - start_time

print(f"\nEntrenamiento completado en {elapsed_time:.1f} segundos")
print(f"Tiempo por época: {elapsed_time/EPOCHS:.1f} segundos")

## 9. Evaluación en Conjunto de Test

In [ ]:
y_test_pred = net.forward(X_test)
y_pred_labels = np.argmax(y_test_pred, axis=1)

accuracy = np.mean(y_test == y_pred_labels) * 100

print("\n" + "="*60)
print("RESULTADOS EN CONJUNTO DE TEST")
print("="*60)
print(f"")
print(f"  Precisión (Accuracy): {accuracy:.2f}%")
print(f"  Muestras correctas:   {np.sum(y_test == y_pred_labels):,} / {len(y_test):,}")
print(f"  Muestras incorrectas: {np.sum(y_test != y_pred_labels):,}")
print(f"")

if accuracy >= 80:
    print("✅ Se cumple el requisito mínimo de ≥80% accuracy")
else:
    print("⚠️ No se alcanza el 80%, considerar más épocas o ajustar hiperparámetros")

## 10. Curvas de Pérdida

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-', linewidth=2, label='Entrenamiento')
axes[0].plot(range(1, len(val_losses)+1), val_losses, 'r--', linewidth=2, label='Validación')
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('Pérdida (Cross-Entropy)', fontsize=12)
axes[0].set_title('Curvas de Pérdida - MNIST', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(train_losses)+1), train_losses, 'b-', linewidth=2, label='Entrenamiento')
axes[1].plot(range(1, len(val_losses)+1), val_losses, 'r--', linewidth=2, label='Validación')
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('Pérdida (escala log)', fontsize=12)
axes[1].set_title('Curvas de Pérdida - Escala Logarítmica', fontsize=14)
axes[1].set_yscale('log')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../memoria/fig_loss_mnist.png', dpi=150)
plt.close()

print("Gráfica guardada en: memoria/fig_loss_mnist.png")
print(f"\nPérdida inicial (train): {train_losses[0]:.4f}")
print(f"Pérdida final (train):   {train_losses[-1]:.4f}")
print(f"Pérdida final (val):     {val_losses[-1]:.4f}")
print(f"Reducción de pérdida:    {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 11. Matriz de Confusión

In [ ]:
num_classes = 10
confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int32)

for true, pred in zip(y_test, y_pred_labels):
    confusion_matrix[true, pred] += 1

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(confusion_matrix, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(xticks=np.arange(num_classes),
       yticks=np.arange(num_classes),
       xticklabels=[str(i) for i in range(10)],
       yticklabels=[str(i) for i in range(10)],
       ylabel='Dígito Real',
       xlabel='Dígito Predicho',
       title='Matriz de Confusión - MNIST')

thresh = confusion_matrix.max() / 2.
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, format(confusion_matrix[i, j], 'd'),
                ha="center", va="center",
                color="white" if confusion_matrix[i, j] > thresh else "black",
                fontsize=9)

plt.tight_layout()
plt.savefig('../memoria/fig_confusion_mnist.png', dpi=150)
plt.close()

print("Gráfica guardada en: memoria/fig_confusion_mnist.png")

## 12. Precisión por Dígito

In [ ]:
print("\nPrecisión por dígito:")
print("="*50)
print(f"{'Dígito':<10} {'Correctos':<12} {'Total':<10} {'Accuracy':<10}")
print("-"*50)

per_class_accuracy = []
for digit in range(10):
    mask = y_test == digit
    correct = np.sum(y_pred_labels[mask] == digit)
    total = np.sum(mask)
    acc = correct / total * 100
    per_class_accuracy.append(acc)
    print(f"{digit:<10} {correct:<12} {total:<10} {acc:.2f}%")

print("-"*50)
print(f"{'Promedio':<10} {'':<12} {'':<10} {np.mean(per_class_accuracy):.2f}%")

plt.figure(figsize=(10, 5))
colors = ['green' if acc >= 90 else 'orange' if acc >= 80 else 'red' for acc in per_class_accuracy]
plt.bar(range(10), per_class_accuracy, color=colors, edgecolor='black')
plt.axhline(y=80, color='r', linestyle='--', label='Umbral mínimo (80%)')
plt.axhline(y=90, color='g', linestyle='--', label='Objetivo (90%)')
plt.xlabel('Dígito', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Precisión por Dígito - MNIST', fontsize=14)
plt.xticks(range(10))
plt.ylim(0, 105)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../memoria/fig_accuracy_per_digit.png', dpi=150)
plt.close()

print("\nGráfica guardada en: memoria/fig_accuracy_per_digit.png")

## 13. Visualización de Errores

Analizamos algunos casos donde la red se equivoca para entender sus limitaciones.

In [ ]:
error_indices = np.where(y_test != y_pred_labels)[0]

print(f"Total de errores: {len(error_indices)}")
print(f"\nMostrando algunos ejemplos mal clasificados:")

n_errors_to_show = min(20, len(error_indices))
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('Ejemplos Mal Clasificados', fontsize=14)

for i in range(n_errors_to_show):
    idx = error_indices[i]
    ax = axes[i // 10, i % 10]
    img = X_test[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'R:{y_test[idx]} P:{y_pred_labels[idx]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../memoria/fig_mnist_errors.png', dpi=150)
plt.close()

print("Gráfica guardada en: memoria/fig_mnist_errors.png")

print("\nErrores más comunes (Real → Predicho):")
error_pairs = {}
for idx in error_indices:
    pair = (y_test[idx], y_pred_labels[idx])
    error_pairs[pair] = error_pairs.get(pair, 0) + 1

sorted_errors = sorted(error_pairs.items(), key=lambda x: x[1], reverse=True)[:10]
for (real, pred), count in sorted_errors:
    print(f"  {real} → {pred}: {count} errores")

## 14. Resumen de Resultados

In [ ]:
print("\n" + "="*70)
print("RESUMEN DEL EXPERIMENTO MNIST")
print("="*70)
print(f"")
print(f"| Métrica                      | Valor                    |")
print(f"|------------------------------|--------------------------|")
print(f"| Dataset                      | MNIST                    |")
print(f"| Muestras de entrenamiento    | {len(X_train):,}                  |")
print(f"| Muestras de validación       | {len(X_val):,}                   |")
print(f"| Muestras de test             | {len(X_test):,}                  |")
print(f"| Arquitectura                 | 784-128-64-10            |")
print(f"| Activaciones                 | ReLU + Softmax           |")
print(f"| Inicialización               | He (ocultas) + Xavier    |")
print(f"| Optimizador                  | Adam (lr={LEARNING_RATE})         |")
print(f"| Épocas                       | {EPOCHS}                       |")
print(f"| Batch Size                   | {BATCH_SIZE}                       |")
print(f"| Tiempo de entrenamiento      | {elapsed_time:.1f} segundos           |")
print(f"| Pérdida inicial (train)      | {train_losses[0]:.4f}                   |")
print(f"| Pérdida final (train)        | {train_losses[-1]:.4f}                   |")
print(f"| Pérdida final (val)          | {val_losses[-1]:.4f}                   |")
print(f"| **Accuracy en Test**         | **{accuracy:.2f}%**               |")
print(f"")

## 15. Conclusiones del Experimento MNIST

### Resultados obtenidos:

1. **Convergencia demostrada:** Las curvas de pérdida muestran una disminución consistente durante el entrenamiento, sin signos de divergencia.

2. **Precisión alcanzada:** Se supera el umbral mínimo requerido del 80% de accuracy en el conjunto de test.

3. **Generalización:** La diferencia entre pérdida de entrenamiento y validación indica un nivel aceptable de generalización sin overfitting severo.

### Validación de la implementación:

Este experimento demuestra que el motor de redes neuronales:

1. ✅ **Escala a datasets grandes** (60,000+ muestras)
2. ✅ **Maneja alta dimensionalidad** (784 features de entrada)
3. ✅ **Soporta redes profundas** (múltiples capas ocultas)
4. ✅ **El optimizador Adam converge** en problemas reales
5. ✅ **Los mini-batches funcionan** correctamente con datasets grandes
6. ✅ **Las funciones de activación ReLU** se comportan adecuadamente

### Análisis de errores:

Los errores más comunes ocurren entre dígitos visualmente similares (por ejemplo, 4↔9, 3↔5, 7↔1), lo cual es esperado y coherente con las dificultades típicas de este dataset.

### Posibles mejoras:

- Aumentar el número de épocas
- Añadir regularización (L2, dropout)
- Usar learning rate scheduling
- Experimentar con arquitecturas más profundas